#### PART 2 — DECODER (GPT)

##### What GPT is
- The Decoder half of the Transformer.
- Reads text strictly left to right.
- Built for generating text, not just understanding it.

##### Why GPT is unidirectional
- Reads left to right only, one word at a time; never looks at future words.
- Example: "My credit card payment is ..." → predicts "overdue" using only prior words.
- This causal reading style is why GPT is great at generating text.

##### GPT vs BERT, key differences
- Direction: BERT bidirectional; GPT unidirectional (left→right).
- Sees: BERT sees all words at once; GPT sees only previous words.
- Task: BERT = understanding; GPT = generating.
- Attention type: BERT = full self-attention; GPT = masked/causal self-attention.
- Pretraining: BERT = MLM + NSP; GPT = next-token prediction (causal language modeling).
- Special tokens: BERT has [CLS]; GPT has none.
- Typical use: BERT → intent classification, NER; GPT → answer generation, chatbots.
- Architecture: GPT has no encoder and no cross-attention — purely stacked decoder blocks.

##### GPT model versions, quick reference
- GPT-1: 12 layers, 768 hidden, 12 heads, ~117M params (same shape as BERT-base; differs in attention type + pretraining objective).
- GPT-2: 48 layers, 1600 hidden, 25 heads, ~1.5B params.
- GPT-3: 96 layers, 12288 hidden, 96 heads, ~175B params.
- GPT-4: undisclosed.
- GPT-2 family depth: Small = 12 layers, Medium = 24, Large = 36, XL = 48; GPT-3 (base) = 96.

##### What "parameters" means
- Learnable weights/biases adjusted during training.
- GPT-3 has 175 billion of them, each a small number like 0.0023 or −1.47.
- A query fires all 175B numbers like a voting system; the combined result emerges as the answer (e.g. "11%").
- Analogy: no single brain cell stores your mother's face; no single number stores "11%" — it emerges from billions working together.

##### Steps in Decoder
Step 1: Input Embedding
- Example query: "What is the Bajaj Finance Fixed Deposit interest rate for senior citizens?"
- Each token → dense vector (768-D in GPT-2 small, same convention as BERT).
- Captures domain relationships: "FD" ↔ "rate" ↔ "maturity."
- Example: "Bajaj" → [0.77, -0.44, 0.61, ...], "rate" → [0.48, -0.25, 0.62, ...].

Step 2: Positional Encoding
- Learned positional embeddings, same general approach as BERT.
- Final input = Token Embedding + Positional Embedding.
- Max sequence length: GPT-2 = 1024 tokens; GPT-3 = 2048 tokens.
- Key difference from BERT: no segment embedding at all in GPT.

Step 3: Masked Self-Attention (Causal Attention)
- Each token attends to itself + all previous tokens, never future tokens.
- Example sequence [Bajaj, Finance, FD, interest, rate]:
    - Bajaj → can see: Bajaj only.
    - Finance → can see: Bajaj, Finance.
    - FD → can see: Bajaj, Finance, FD.
    - interest → can see: Bajaj, Finance, FD, interest.
    - rate → can see: all 5 (nothing left to block).
- Mask sets future positions to −∞ before softmax; softmax(−∞) = 0 → fully blocked.
- Formula: Attention(Q,K,V) = softmax(QKᵗ/√d_k + Mask) · V. Mask = 0 (allowed) or −∞ (future).
- Worked attention weights for "rate" looking back: interest = 0.42 (core phrase), FD = 0.28 (product), Finance = 0.16 (entity), Bajaj = 0.11 (brand), is = 0.03 (low impact).
- Result: "rate" now effectively means "Bajaj Finance FD interest rate," not loan rate, not generic rate.
- Q/K/V meaning for token "rate": Query = what it's looking for from the past; Key = what each past token offers; Value = the actual content contributed.


Step 4: Multi-Head Masked Attention
- Multiple heads, each learning a different left-context relationship, all respecting the causal mask.
- GPT-2 small: 12 heads. Example: head 1 = product ID ("Fixed Deposit"↔"FD"), head 2 = rate/value association, head 3 = time/tenure, head 4 = customer segment, head 5 = brand specificity, heads 6+ = numbers/facts/grammar.
- Math: MultiHead(Q,K,V) = Concat(head_1...head_h)·Wᵒ, where head_i = Attention(Q_i,K_i,V_i) with causal mask applied.


Step 5: Add & Normalize (after attention)
- Output = LayerNorm(input + masked-attention-output).
- GPT-2/3 specific detail: uses Pre-LayerNorm — normalize before attention/FFN, not after (better for deep models).


Step 6: Feedforward Network (FFN)
- Applied independently per position. Expand d_model → 4×d_model (768→3072 in GPT-2 small), GeLU, shrink back.
- Formula: FFN(x) = GeLU(xW₁+b₁)W₂+b₂.
- Worked intuition ("Bajaj Finance"): before FFN = vague (FD-related, maybe loan noise). After FFN = sharp ("Bajaj Finance FD annual rate," noise removed, expects a % value next).


Step 7: Add & Normalize (after FFN)
- Output = LayerNorm(input + FFN-output).
- Preserves attention context + FFN refinements together.


Step 8: Stacked Decoder Layers
- Steps 3–7 repeated across multiple decoder blocks; deeper layers = higher-level understanding.
- Worked example by layer: layer 1 = brand entity + "FD"=Fixed Deposit; layer 3 = "FD interest rate" as product-rate phrase, "senior citizens" as segment; layer 6 = senior citizens get different rates; layer 9 = product inquiry about a specific FD variant; layer 12 = full intent (exact annual rate for senior citizens).


Step 9: Language Model Head (output layer)
- Predicts the next token at every position.
- Final hidden state (768-D) → linear layer → vocab size (~50257 in GPT-2) → logits → softmax → probability distribution.
Worked example, input "...FD interest rate is": "8.60%" logit 9.2 → prob 0.72 (wins); "7.5%" logit 6.1 → prob 0.18; "loan" logit 1.2 → prob 0.07; "hello" logit 0.3 → prob 0.03.


Step 10: Autoregressive Generation (the loop)
- One token generated at a time; each new token is appended back to the input; process repeats.
- Worked example: "...rate" → predict "is" → "...rate is" → predict "8.60%" → "...is 8.60%" → predict "per" → continues through "annum," "for," "senior," "citizens," end-of-sequence.
- Final output: "Bajaj Finance FD interest rate is 8.60% per annum for senior citizens."
- Key reminder: GPT only ever sees the past; it generates the future, one token at a time.


##### Decoding/sampling strategies
- Greedy decoding: always picks highest-probability token; simple but can be repetitive.
- Top-k sampling: randomly samples from only the top k most likely tokens.
- Top-p (nucleus) sampling: samples from smallest token set whose combined probability ≥ p.
- Temperature: high = more creative/random; low = more focused/factual (better for exact rate queries).


##### Pre-LN vs Post-LN
- LayerNorm rescales values so they don't get too large/small, keeping training stable.
- Post-LN (BERT, original Transformer): attention/FFN first → add residual → normalize after. "Work first, clean up later."
- Pre-LN (GPT-2/3): normalize first → attention/FFN → add residual after. "Clean first, then work."

Why Pre-LN for GPT-2/3
- Better gradient flow, more stable training for very deep models (48+ layers), enables scaling to very large LLMs.
- Analogy: Post-LN = teacher checks work only after the exam; Pre-LN = teacher prepares/stabilizes the student before the exam even starts.
- Email-classification illustration: a 10-page FD email, a short "FD?", and a normal 3-line FD email — without LayerNorm, wildly different sizes confuse the model; with LayerNorm, all rescale to the same range and classify correctly as "FD Query."
- At scale: Post-LN at deep layers → errors accumulate layer by layer → unstable. Pre-LN even at 96 layers → every layer gets clean input first → stable.


##### GPT pretraining: Autoregressive Generation / next-token prediction
- Trained to predict the next token using only previous tokens.
- Worked build-up: "Bajaj"→"Finance"; "Bajaj Finance"→"Personal"; "...Personal"→"Loan"; "...Loan"→"interest"; "...interest"→"rate"; "...rate"→"starts"; "...starts"→"from"; "...from"→"11%."


##### The Causal Mask, mechanically
- Blocks any token from attending to future tokens.
- Example over [Bajaj(0), Finance(1), personal(2), loan(3), rate(4)]: each position can attend only to itself and earlier positions (Bajaj sees only Bajaj; rate sees all 5).
- Mask value = 0 for allowed positions, −∞ for future positions; softmax(−∞) = 0, fully zeroing out future attention.


##### RLHF (Reinforcement Learning from Human Feedback)
- Transforms a raw pretrained GPT into a genuinely helpful assistant (e.g. ChatGPT-style).
- Raw GPT only predicts the statistically next token. RLHF additionally teaches it to follow instructions, behave safely, and match human preferences.
- Flow: pretrained GPT → humans rank multiple responses → reward model learns preferences → PPO fine-tunes GPT using that reward → final helpful assistant.

Worked example, prompt "How to reply to FD penalty complaint?":
- Without RLHF: "Penalty is charged when FD rules are violated. Charges depend on policy and delay conditions." (cold, statistical)
- With RLHF: "Dear Customer, we regret the inconvenience caused due to the FD penalty charges. We understand your concern and request you to share your FD details so we can review the case and assist you further." (helpful, polite)

What RLHF teaches: helpfulness, instruction-following, politeness, safer responses.
- Interview point: pretraining = language patterns; RLHF = alignment with human preferences — two different goals.
- Simple intuition: raw GPT = knowledgeable new employee; RLHF = the customer-service training on how to actually interact with people.

##### Important interview points on GPT
- Decoder-only: no encoder, no cross-attention.
- Uses learned positional embeddings (like BERT).
- Uses a causal mask to block future tokens.
- Pre-LN in GPT-2/3 improves training stability at scale (vs BERT's Post-LN).
- Simplest summary: BERT understands and classifies; GPT generates.

GPT summary flow (high level):
Text → Tokenize → Embed + Position → Masked Multi-Head Attention → Add & Norm → Feedforward Network → Add & Norm → repeat across L decoder layers → LM Head → predict next token → append → repeat until generation is complete.